# Long Context Reorder

- Author: [Minji](https://github.com/r14minji)
- Peer Review: 
- This is a part of [LangChain OpenTutorial](https://github.com/LangChain-OpenTutorial/LangChain-OpenTutorial)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LangChain-OpenTutorial/LangChain-OpenTutorial/blob/main/02-Prompt/02-FewShotPromptTemplate.ipynb) [![Open in GitHub](https://img.shields.io/badge/Open%20in%20GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/LangChain-OpenTutorial/LangChain-OpenTutorial/blob/main/02-Prompt/02-FewShotPromptTemplate.ipynb)


## Overview

Regardless of the model's architecture, performance significantly degrades when including more than 10 retrieved documents.

Simply put, when the model needs to access relevant information in the middle of a long context, it tends to ignore the provided documents.

For more details, please refer to the following paper:

- https://arxiv.org/abs/2307.03172

To avoid this issue, you can prevent performance degradation by reordering documents after retrieval.

Create a retriever that can store and search text data using the Chroma vector store.
Use the retriever's invoke method to search for highly relevant documents for a given query.


### Table of Contents

- [Overview](#overview)
- [Environment Setup](#environment-setup)
- [Create an instance of the LongContextReorder class named reordering](#create-an-instance-of-the-longcontextreorder-class-named-reordering)
- [Creating Question-Answering Chain with Context Reordering](#creating-question-answering-chain-with-context-reordering)

---


## Environment Setup

Set up the environment. You may refer to [Environment Setup](https://wikidocs.net/257836) for more details.

**[Note]**
- `langchain-opentutorial` is a package that provides a set of easy-to-use environment setup, useful functions and utilities for tutorials. 
- You can checkout the [`langchain-opentutorial`](https://github.com/LangChain-OpenTutorial/langchain-opentutorial-pypi) for more details.

In [1]:
%%capture --no-stderr
!pip install langchain-opentutorial

In [3]:
# Configuration file for managing API keys as environment variables
from dotenv import load_dotenv

# Load API key information
load_dotenv(override=True)

True

In [4]:

from langchain_opentutorial import package

package.install(
    [
       "langsmith",
        "langchain",
        "langchain_openai",
        "langchain_community",
        "langchain-chroma",
    ],
    verbose=False,
    upgrade=False,
)


[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from langchain_opentutorial import set_env

set_env(
    {
        # "OPENAI_API_KEY": "",
        "LANGCHAIN_API_KEY": "",
        "LANGCHAIN_TRACING_V2": "true",
        "LANGCHAIN_ENDPOINT": "https://api.smith.langchain.com",
        "LANGCHAIN_PROJECT": "04-LongContextReorder",
    }
)

Environment variables have been set successfully.


## Create an instance of the LongContextReorder class named reordering.

Enter a query for the retriever to perform the search.

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_community.document_transformers import LongContextReorder
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

# Get embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

texts = [
    "나는 사과를 좋아해!",
    "애플은 사과 사과는 애플",
    "애플에서 구매한 맥북 너무 좋다",
    "애플에서 애플카 출시 한다던데?",
    "영등포 청과 시장에서 판매하는 사과는 맛있어",
    "애플 워치와 에어팟 같은 웨어러블 기기도 애플의 인기 제품군에 속합니다.",
    "아침에 사과는 좋은 습관이야",
    "이번에 관악산 등산갈때 사과를 가져갈까?",
    "내가 저번에 실수를 해서 사과를 해야할거같아",
    "사과를 했더니 받아줘서 다행이야",
]



# 검색기 생성 (Set K to 10)
retriever = Chroma.from_texts(texts, embedding=embeddings).as_retriever(
    search_kwargs={"k": 10}
)

In [7]:

query = "사과의 대해 말해줄수 있어??"

# Retrieves relevant documents sorted by relevance score.
docs = retriever.invoke(query)
'''
[ 출력 결과 ]
[Document(metadata={}, page_content='사과를 했더니 받아줘서 다행이야'),
 Document(metadata={}, page_content='내가 저번에 실수를 해서 사과를 해야할거같아'),
 Document(metadata={}, page_content='나는 사과를 좋아해!'),
 Document(metadata={}, page_content='이번에 관악산 등산갈때 사과를 가져갈까?'),
 Document(metadata={}, page_content='애플은 사과 사과는 애플'),
 Document(metadata={}, page_content='아침에 사과는 좋은 습관이야'),
 Document(metadata={}, page_content='영등포 청과 시장에서 판매하는 사과는 맛있어'),
 Document(metadata={}, page_content='애플에서 구매한 맥북 너무 좋다'),
 Document(metadata={}, page_content='애플에서 애플카 출시 한다던데?'),
 Document(metadata={}, page_content='애플 워치와 에어팟 같은 웨어러블 기기도 애플의 인기 제품군에 속합니다.')]
'''



Create an instance of LongContextReorder class.

- Call reordering.transform_documents(docs) to reorder the document list.
- Less relevant documents are positioned in the middle of the list, while more relevant documents are positioned at the beginning and end.


In [7]:
# 문서를 재정렬합니다
# 덜 관련된 문서는 목록의 중간에 위치하고 더 관련된 요소는 시작/끝에 위치합니다.

reordering = LongContextReorder()
reordered_docs = reordering.transform_documents(docs)

# Verify that 4 relevant documents are positioned at start and end
reordered_docs


'''
[ 출력 결과 ]
[Document(metadata={}, page_content='내가 저번에 실수를 해서 사과를 해야할거같아'),
 Document(metadata={}, page_content='이번에 관악산 등산갈때 사과를 가져갈까?'),
 Document(metadata={}, page_content='아침에 사과는 좋은 습관이야'),
 Document(metadata={}, page_content='애플에서 구매한 맥북 너무 좋다'),
 Document(metadata={}, page_content='애플 워치와 에어팟 같은 웨어러블 기기도 애플의 인기 제품군에 속합니다.'),
 Document(metadata={}, page_content='애플에서 애플카 출시 한다던데?'),
 Document(metadata={}, page_content='영등포 청과 시장에서 판매하는 사과는 맛있어'),
 Document(metadata={}, page_content='애플은 사과 사과는 애플'),
 Document(metadata={}, page_content='나는 사과를 좋아해!'),
 Document(metadata={}, page_content='사과를 했더니 받아줘서 다행이야')]
'''

[Document(metadata={}, page_content='내가 저번에 실수를 해서 사과를 해야할거같아'),
 Document(metadata={}, page_content='이번에 관악산 등산갈때 사과를 가져갈까?'),
 Document(metadata={}, page_content='아침에 사과는 좋은 습관이야'),
 Document(metadata={}, page_content='애플에서 구매한 맥북 너무 좋다'),
 Document(metadata={}, page_content='애플 워치와 에어팟 같은 웨어러블 기기도 애플의 인기 제품군에 속합니다.'),
 Document(metadata={}, page_content='애플에서 애플카 출시 한다던데?'),
 Document(metadata={}, page_content='영등포 청과 시장에서 판매하는 사과는 맛있어'),
 Document(metadata={}, page_content='애플은 사과 사과는 애플'),
 Document(metadata={}, page_content='나는 사과를 좋아해!'),
 Document(metadata={}, page_content='사과를 했더니 받아줘서 다행이야')]

## Creating Question-Answering Chain with Context Reordering

A chain that enhances QA (Question-Answering) performance by reordering documents using LongContextReorder, which optimizes the arrangement of context for better comprehension and response accuracy.

In [11]:
def format_docs(docs):
    return "\n".join([doc.page_content for i, doc in enumerate(docs)])

In [12]:
print(format_docs(docs))

영등포 청과 시장에서 판매하는 사과는 맛있어
사과를 했더니 받아줘서 다행이야
애플에서 구매한 맥북 너무 좋다
애플 워치와 에어팟 같은 웨어러블 기기도 애플의 인기 제품군에 속합니다.
내가 저번에 실수를 해서 사과를 해야할거같아
나는 사과를 좋아해!
애플은 사과 사과는 애플
이번에 관악산 등산갈때 사과를 가져갈까?
애플에서 애플카 출시 한다던데?
아침에 사과는 좋은 습관이야


In [13]:
def format_docs(docs):
    return "\n".join(
        [
            f"[{i}] {doc.page_content} [source: kosaf1996@naver.com]"
            for i, doc in enumerate(docs)
        ]
    )


def reorder_documents(docs):
    # Reorder
    reordering = LongContextReorder()
    reordered_docs = reordering.transform_documents(docs)
    combined = format_docs(reordered_docs)
    print(combined)
    return combined

Prints the reordered documents.

In [14]:
# Define prompt template
_ = reorder_documents(docs)

[0] 사과를 했더니 받아줘서 다행이야 [source: kosaf1996@naver.com]
[1] 애플 워치와 에어팟 같은 웨어러블 기기도 애플의 인기 제품군에 속합니다. [source: kosaf1996@naver.com]
[2] 나는 사과를 좋아해! [source: kosaf1996@naver.com]
[3] 이번에 관악산 등산갈때 사과를 가져갈까? [source: kosaf1996@naver.com]
[4] 아침에 사과는 좋은 습관이야 [source: kosaf1996@naver.com]
[5] 애플에서 애플카 출시 한다던데? [source: kosaf1996@naver.com]
[6] 애플은 사과 사과는 애플 [source: kosaf1996@naver.com]
[7] 내가 저번에 실수를 해서 사과를 해야할거같아 [source: kosaf1996@naver.com]
[8] 애플에서 구매한 맥북 너무 좋다 [source: kosaf1996@naver.com]
[9] 영등포 청과 시장에서 판매하는 사과는 맛있어 [source: kosaf1996@naver.com]


In [15]:
from langchain.prompts import ChatPromptTemplate
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Define prompt template
template = """Given this text extracts:
{context}

-----
Please answer the following question:
{question}

Answer in the following languages: {language}
"""

# Define prompt
prompt = ChatPromptTemplate.from_template(template)

# Define Chain
chain = (
    {
        "context": itemgetter("question")
        | retriever
        | RunnableLambda(reorder_documents),  # Search context based on question
        "question": itemgetter("question"),  # Extract question
        "language": itemgetter("language"),  # Extract answer language
    }
    | prompt  # Pass values to prompt template
    | ChatOpenAI(model="gpt-4o-mini")  # Pass prompt to language model
    | StrOutputParser()  # Parse model output as string
)


Enter the query in question and language for response.

Check the search results of reordered documents.

In [18]:
answer = chain.invoke(
    {"question": "애플 제품을 알려줘", "language": "Korean"}
)

[0] 애플은 사과 사과는 애플 [source: kosaf1996@naver.com]
[1] 애플에서 구매한 맥북 너무 좋다 [source: kosaf1996@naver.com]
[2] 나는 사과를 좋아해! [source: kosaf1996@naver.com]
[3] 영등포 청과 시장에서 판매하는 사과는 맛있어 [source: kosaf1996@naver.com]
[4] 아침에 사과는 좋은 습관이야 [source: kosaf1996@naver.com]
[5] 이번에 관악산 등산갈때 사과를 가져갈까? [source: kosaf1996@naver.com]
[6] 사과를 했더니 받아줘서 다행이야 [source: kosaf1996@naver.com]
[7] 내가 저번에 실수를 해서 사과를 해야할거같아 [source: kosaf1996@naver.com]
[8] 애플 워치와 에어팟 같은 웨어러블 기기도 애플의 인기 제품군에 속합니다. [source: kosaf1996@naver.com]
[9] 애플에서 애플카 출시 한다던데? [source: kosaf1996@naver.com]


Prints the response.

In [14]:
print(answer)

ChatGPT is an AI language model developed by OpenAI, designed to assist users by generating human-like text responses based on the input it receives. It can engage in conversations, answer questions, provide explanations, and generate creative content across various topics. ChatGPT is commonly used for applications such as customer support, content creation, education, and more. It is trained on a diverse dataset, allowing it to understand and produce text in multiple languages and styles. However, it is important to note that while ChatGPT is capable of providing information, it does not have access to real-time data or personal experiences and should not be relied upon for critical decision-making.
